In [3]:
import pandas as pd
import numpy as np
from nelson_siegel_svensson.calibrate import calibrate_ns_ols 
import matplotlib.pyplot as plt
import calendar

/Users/nidhi/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
def tpv(fv, coupon, years, discount_rate):
    # fv: Face value of the bond
    # coupon: Coupon rate in currency units (not percentage)
    # years: Years to maturity
    # discount_rate: Discounting rate in decimal

    if coupon > 0:
        tpv = sum(coupon / ((1 + discount_rate) ** year) for year in range(1, int(years) + 1))
        tpv += fv / ((1 + discount_rate) ** years)
    else:
        tpv = fv / ((1 + discount_rate) ** years)
    return tpv


In [5]:
def ytm(p, cr, y, fv=100):
    # p: current price
    # cr: coupon rate in percent
    # y: years to maturity
    cr = cr / 100.0
    coupon = fv * cr
    guess = cr  # initial guess
    precision = 1e-6
    max_iter = 10000
    for _ in range(max_iter):
        pv = tpv(fv, coupon, y, guess)
        diff = pv - p
        if abs(diff) < precision:
            break
        guess += 0.00001 if diff < 0 else -0.00001
    return guess


In [6]:
# Load bond data
BondDataFilePath = "01-01-2020-TO-30-12-2020GS.csv"
df = pd.read_csv(BondDataFilePath)

# Filter data
df = df[df["Filter"] == 1].copy()
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year
df["Maturity Year"] = df["Security Name"].str[2:6].astype(int)
df["Res Maturity"] = df["Maturity Year"] - df["Year"]
df["Res Maturity"].replace(0, 1, inplace=True)
df["Coupon Rate"] = df["Issue Name"].str.replace("%", "").astype(float)
df = df[["Date", "Month", "LTP", "Res Maturity", "Coupon Rate"]]
df["Res Maturity"] = df["Res Maturity"].astype(float)

# Estimate yield curves
months = df["Month"].unique()
for month in months:
    df_month = df[df["Month"] == month].drop_duplicates()
    df_month["Calc YTM"] = df_month.apply(
        lambda row: ytm(row["LTP"], row["Coupon Rate"], row["Res Maturity"]), axis=1
    )

    time = df_month["Res Maturity"].to_numpy()
    yields = df_month["Calc YTM"].to_numpy()

    curve, status = calibrate_ns_ols(time, yields, 1.0)

    month_name = calendar.month_name[month]
    year_str = df_month.iloc[0]["Date"].year

    t = np.linspace(1, 30, 60)
    fig, ax = plt.subplots()
    ax.plot(t, curve(t))
    plt.xlim(0, 30)
    plt.ylim(0.03, 0.12)
    plt.title(f"Yield Curve for {month_name} {year_str}")
    plt.xlabel("Year")
    plt.ylabel("Yield")
    fig.savefig(f"{month}.png")
    plt.close(fig)

FileNotFoundError: [Errno 2] No such file or directory: '01-01-2020-TO-30-12-2020GS.csv'